[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobinL/uk_address_matcher/blob/main/match_example_data.ipynb)


In [12]:
!pip install --pre uk_address_matcher



zsh:1: command not found: pip


In [13]:
import duckdb
import pandas as pd

from uk_address_matcher.post_linkage.analyse_results import (
    best_matches_summary,
)
from uk_address_matcher.post_linkage.identify_distinguishing_tokens import (
    improve_predictions_using_distinguishing_tokens,
)
from uk_address_matcher import clean_data_with_term_frequencies, get_linker
import time

pd.options.display.max_colwidth = 1000

pd.options.display.max_colwidth = 1000

# -----------------------------------------------------------------------------
# Step 1: Load in some example data.  If using your own data, it must be in
# the same format as the example data.
# -----------------------------------------------------------------------------
# Any additional columns should be retained as-is by the cleaning code

p_fhrs = "https://github.com/RobinL/uk_address_matcher/raw/main/example_data/fhrs_addresses_sample.parquet"
p_ch = "https://github.com/RobinL/uk_address_matcher/raw/main/example_data/companies_house_addresess_postcode_overlap.parquet"

con = duckdb.connect(database=":memory:")
con.sql(f"CREATE TABLE df_fhrs AS SELECT * FROM read_parquet('{p_fhrs}')")
con.sql(f"CREATE TABLE df_ch AS SELECT * FROM read_parquet('{p_ch}')")
df_fhrs = con.table("df_fhrs")
df_ch = con.table("df_ch")

# Display length of the dataset
print(f"Length of FHRS dataset: {len(df_fhrs.df()):,.0f}")
print(f"Length of Companies House dataset: {len(df_ch.df()):,.0f}")

df_fhrs.limit(5).show(max_width=500)
df_ch.limit(5).show(max_width=500)


Length of FHRS dataset: 5,000
Length of Companies House dataset: 21,952
┌───────────┬────────────────┬────────────────────────────────────────────────────────────────────────────────┬──────────┐
│ unique_id │ source_dataset │                                 address_concat                                 │ postcode │
│  varchar  │    varchar     │                                    varchar                                     │ varchar  │
├───────────┼────────────────┼────────────────────────────────────────────────────────────────────────────────┼──────────┤
│ 1543406   │ fhrs           │ 1 OAK CHILTON DAY CENTRE UNIT 2 MARTINS ROAD CHILTON INDUSTRIAL ESTATE SUDBURY │ CO10 2FT │
│ 1395196   │ fhrs           │ 38 STATION ROAD SUDBURY SUFFOLK                                                │ CO10 2SS │
│ 1394874   │ fhrs           │ 33 SWAN STREET BOXFORD SUDBURY SUFFOLK                                         │ CO10 5NZ │
│ 1649158   │ fhrs           │ 11A FRIARS STREET SUDBURY SUFFOLK   

In [14]:
# -----------------------------------------------------------------------------
# Step 2: Clean the data/feature engineering to prepare for matching model
# -----------------------------------------------------------------------------

df_fhrs_clean = clean_data_with_term_frequencies(df_fhrs, con=con)
df_ch_clean = clean_data_with_term_frequencies(df_ch, con=con)


In [15]:
linker = get_linker(
    df_addresses_to_match=df_fhrs_clean,
    df_addresses_to_search_within=df_ch_clean,
    con=con,
    include_full_postcode_block=True,
    additional_columns_to_retain=["original_address_concat"],
    retain_intermediate_calculation_columns=True,
)

df_predict = linker.inference.predict(
    threshold_match_weight=-50
)
df_predict_ddb = df_predict.as_duckdbpyrelation()

Blocking time: 0.10 seconds
Predict time: 1.01 seconds


In [16]:
start_time = time.time()
df_predict_improved = improve_predictions_using_distinguishing_tokens(
    df_predict=df_predict_ddb,
    con=con,
    match_weight_threshold=-20,
)

df_predict_improved.show(max_width=500, max_rows=5)

end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

┌─────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────────────────┬──────────────────────┬───────────────────┬──────────────────────┬──────────────────────┬─────────────────────┬──────────────────────┬──────────────────────┬────────────────────────────────┬──────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────┬────────────┬─────────────────────────────────────────────────────────────────────┬────────────┐
│ unique_id_l │ unique_id_r │  ukam_address_id_r   │  ukam_address_id_l   │    mw_adjustment    │ match_weight_origi…  │   match_weight    │ overlapping_tokens…  │ tokens_elsewhere_i…  │   missing_tokens    │ overlapping_bigram…  │ bigrams_elsewhere_…  │ overlapping_bigrams_this_l_a…  │ bigrams_elsewhere_in_block_but_not_this_filtered │                    original_address_concat_l                    │ postcode_l │                      original_address_concat_r                      │ postcode_r 

In [17]:
print("\nResults before second pass:")
dsum_1 = best_matches_summary(
    df_predict=df_predict_ddb, df_addresses_to_match=df_fhrs_clean, con=con
)
dsum_1.show(max_width=500, max_rows=20)

print("\nResults after second pass:")
dsum_2 = best_matches_summary(
    df_predict=df_predict_improved, df_addresses_to_match=df_fhrs_clean, con=con
)
dsum_2.show(max_width=500, max_rows=20)



Results before second pass:
┌─────────────────────────────┬───────┬────────────┐
│ distinguishability_category │ count │ percentage │
│           varchar           │ int64 │  varchar   │
├─────────────────────────────┼───────┼────────────┤
│ 01: One match only          │   749 │ 14.98%     │
│ 02: Distinguishability > 10 │   662 │ 13.24%     │
│ 03: Distinguishability > 5  │   169 │ 3.38%      │
│ 04: Distinguishability > 1  │   520 │ 10.40%     │
│ 05: Distinguishability > 0  │    74 │ 1.48%      │
│ 06.: Distinguishability = 0 │  1726 │ 34.52%     │
│ 99: No match                │  1100 │ 22.00%     │
└─────────────────────────────┴───────┴────────────┘


Results after second pass:
┌─────────────────────────────┬───────┬────────────┐
│ distinguishability_category │ count │ percentage │
│           varchar           │ int64 │  varchar   │
├─────────────────────────────┼───────┼────────────┤
│ 01: One match only          │   749 │ 14.98%     │
│ 02: Distinguishability > 10 │   871 │ 1

/Users/thomashepworth/Data_Engineering/work_projects/uk_address_matcher_worktrees/uk_address_matcher__main/uk_address_matcher/post_linkage/analyse_results.py:82: UserWarning: 
Most users will wish to pass the result of improve_predictions_using_distinguishing_tokens to this function.
You appear to have passed the raw output of linker.inference.predict.
  warnings.warn(
